# ShelfTwin 02: pack + gap detector (resumable)

Fine-tunes the notebook-01 detector to two classes, **pack** and **gap**, on 640 px square crops. Their scale matches deployment,
where a 1080p frame is split into two ~1080 px tiles and each is run at 640.

* *Real gaps* (all splits): geometric heuristic on real photos (a wide space between neighbouring packs in a shelf row).
* *Synthetic gaps* (**train only**, spec 5.2.7): runs of packs erased and filled by inpainting, shelf-back texture, or real gap patches.

Metrics: per-class AP, and **gap recall at precision 0.9** on real test photos.

**Before running (Kaggle notebook settings, right-hand panel):**
1. *Accelerator*: **GPU T4 x2** · *Internet*: **On** (phone-verified account)
2. *Add Input*: the SKU-110K dataset **and** the output of your notebook-01 run (*Your Work → Notebooks → ShelfTwin 01*): its weights are used to start from, and its prepared dataset cache is reused, which skips about 10 min of preparation
3. First run: set `SMOKE_TEST = True` and *Run All* (~15 min). It must end with `ALL DONE`. Then set it back to `False`.
4. Real run: **Save Version → Save & Run All (Commit)**. It runs in the background and keeps `/kaggle/working` as output.

**Resuming (built in).** If the run prints `NOT FINISHED`, open the notebook again, *Add Input → Your Work → Notebooks →*
this notebook (its latest version), and *Save & Run All* again. Checkpoints, the prepared dataset cache and the
training state are restored automatically and training continues from the last epoch with the same schedule.

Outputs: `outputs/pack_gap.pt` (+ `.onnx` when finished), `outputs/metrics_02.json`.

In [ ]:
SMOKE_TEST = False
SESSION_HOURS = 11.0
RESERVE_H = 0.6
EPOCHS = 30
SRC_SIDE = 896              # must match notebook 01's IMGSZ to reuse its dataset cache
MAX_TRAIN_IMAGES = 5000     # source photos for training (x2 crops each); None = all ~8.2k
IMGSZ = 640
BATCH = 32
BATCH_PER_GPU = 16

In [ ]:
import os, sys
os.makedirs("/kaggle/working/src", exist_ok=True)
sys.path.insert(0, "/kaggle/working/src")

import os, sys, subprocess
def _ultra_ok():
    # importlib.metadata does not import the package, so an old version never gets cached in this kernel
    from importlib.metadata import version, PackageNotFoundError
    try:
        s = version("ultralytics")
    except PackageNotFoundError:
        return False, None
    try:
        v = tuple(int("".join(ch for ch in x if ch.isdigit()) or 0) for x in s.split(".")[:3])
    except Exception:
        return False, s
    return v >= (8, 4, 134), s
# Ultralytics >= 8.4.134 retries dense-image label assignment per image on GPU after an OOM
# instead of a very slow CPU fallback (SKU-110K has 150-700 boxes per image).
ok, ver = _ultra_ok()
print("ultralytics installed:", ver)
if not ok:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics>=8.4.134"])
    if r.returncode != 0:
        print("WARNING: could not install ultralytics>=8.4.134 (Internet on?). Trying any version.")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

print(subprocess.run([sys.executable, "-c", "import ultralytics, torch; print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| GPUs', torch.cuda.device_count())"], capture_output=True, text=True).stdout)

In [ ]:
%%writefile /kaggle/working/src/kaggle_common.py
"""Shared helpers for ShelfTwin Kaggle notebooks (embedded into each notebook).

Everything here is defensive: no hardcoded dataset paths, clear errors, and
environment fixes for known Kaggle pitfalls (W&B login prompts, NCCL hangs).
"""
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("ST_INPUT_ROOT", "/kaggle/input"))
WORK = Path(os.environ.get("ST_WORK", "/kaggle/working"))
TMP = Path(os.environ.get("ST_TMP", "/tmp/shelftwin"))
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SESSION_START = time.time()


def log(*a):
    print(time.strftime("[%H:%M:%S]"), *a, flush=True)


def quiet_loggers():
    """Ultralytics auto-enables W&B/Comet/ClearML/MLflow/DVC when installed; on Kaggle a
    W&B login prompt can block a non-interactive run. Disable all of them."""
    os.environ["WANDB_MODE"] = "disabled"
    os.environ["WANDB_DISABLED"] = "true"
    os.environ["COMET_MODE"] = "disabled"
    os.environ["CLEARML_OFF"] = "1"
    # Known T4 x2 NCCL peer-to-peer hangs in containers; harmless when single GPU
    os.environ.setdefault("NCCL_P2P_DISABLE", "1")
    os.environ.setdefault("NCCL_IB_DISABLE", "1")
    try:
        from ultralytics import settings
        settings.update({k: False for k in ("wandb", "comet", "clearml", "mlflow", "dvc",
                                            "neptune", "raytune", "tensorboard", "hub")
                         if k in settings})
    except Exception as e:  # ultralytics not installed yet or older settings keys
        log("note: could not update ultralytics settings:", e)


def pip_install(*pkgs: str, quiet: bool = True) -> bool:
    cmd = [sys.executable, "-m", "pip", "install"] + (["-q"] if quiet else []) + list(pkgs)
    r = subprocess.run(cmd)
    if r.returncode != 0:
        log("pip install failed for", pkgs, "- continuing; check that Internet is ON in notebook settings")
    return r.returncode == 0


def gpu_info() -> list[dict]:
    try:
        import torch
        return [dict(i=i, name=torch.cuda.get_device_name(i),
                     mem_gb=round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1))
                for i in range(torch.cuda.device_count())]
    except Exception:
        return []


def disk_free_gb(path: Path | str = "/") -> float:
    p = Path(path)
    while not p.exists() and p != p.parent:
        p = p.parent
    return shutil.disk_usage(p).free / 2**30


def elapsed_h() -> float:
    return (time.time() - SESSION_START) / 3600


def walk_limited(root: Path, max_depth: int = 6):
    """os.walk with a depth limit; yields (dirpath, dirnames, filenames)."""
    root = Path(root)
    if not root.exists():
        return
    base = len(root.parts)
    for dp, dns, fns in os.walk(root, followlinks=True):
        depth = len(Path(dp).parts) - base
        if depth >= max_depth:
            dns[:] = []
        dns.sort()
        yield dp, dns, fns


def find_files(name_pred, root: Path = INPUT_ROOT, max_depth: int = 7, limit: int = 1000) -> list[Path]:
    out = []
    for dp, _, fns in walk_limited(root, max_depth):
        for f in fns:
            if name_pred(f):
                out.append(Path(dp) / f)
                if len(out) >= limit:
                    return out
    return out


def print_input_tree(root: Path = INPUT_ROOT, max_depth: int = 3, max_entries: int = 12):
    """Prints what is attached, so a missing dataset is easy to diagnose."""
    log(f"Contents of {root} (depth {max_depth}):")
    if not Path(root).exists():
        print("  (does not exist)")
        return
    for dp, dns, fns in walk_limited(root, max_depth):
        depth = len(Path(dp).parts) - len(Path(root).parts)
        ind = "  " * depth
        print(f"{ind}{Path(dp).name}/  ({len(dns)} dirs, {len(fns)} files)")
        for f in fns[:3]:
            print(f"{ind}  {f}")
        if len(dns) > max_entries:
            dns[:] = dns[:max_entries]


def find_weights(patterns=("best.pt",), roots=(INPUT_ROOT, WORK), must_contain: str | None = None) -> Path | None:
    """Find weights from a previous notebook's output attached as input. Newest wins."""
    cands = []
    for r in roots:
        for p in find_files(lambda f: f in patterns, Path(r), max_depth=9):
            if must_contain is None or must_contain in str(p):
                cands.append(p)
    if not cands:
        return None
    return max(cands, key=lambda p: p.stat().st_mtime)


def save_json(obj, path: Path | str):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=2, default=str))


def _progress_line(raw: bytes) -> str:
    """tqdm redraws a bar with a carriage return; keep only its final state."""
    s = raw.decode("utf-8", errors="replace").rstrip("\r\n ")  # CRLF-terminated lines too
    return s.split("\r")[-1].rstrip()


STOPPED_BY_POLICY = -2


def cpu_workers(reserve: int = 0) -> int:
    return max(1, (os.cpu_count() or 2) - reserve)


def find_cached(name: str) -> Path | None:
    """A cache file produced by an earlier session: in WORK/cache (Colab: Drive) or in an
    attached previous notebook output under INPUT_ROOT (Kaggle)."""
    p = WORK / "cache" / name
    if p.exists():
        return p
    hits = find_files(lambda f: f == name, INPUT_ROOT, max_depth=6, limit=5)
    return hits[0] if hits else None


def cached_dir(name: str, dst: Path, build, keep_cache: bool = True) -> Path:
    """Return `dst`, restoring it from `<name>.tar` if an earlier session cached it, else
    building it with `build(dst)` and caching it to WORK/cache (saved as notebook output on
    Kaggle, on Drive in Colab). Restoring a single tar is much faster than rebuilding."""
    import tarfile
    marker = dst / ".complete"
    if marker.exists():
        log("Using prepared", dst)
        return dst
    tar = find_cached(name + ".tar")
    if tar is not None:
        log(f"Restoring {dst.name} from cache {tar} ...")
        t0 = time.time()
        if dst.exists():
            shutil.rmtree(dst)
        dst.parent.mkdir(parents=True, exist_ok=True)
        try:
            with tarfile.open(tar) as tf:
                try:
                    tf.extractall(dst.parent, filter="data")
                except TypeError:
                    tf.extractall(dst.parent)
            if marker.exists():
                log(f"  restored in {time.time() - t0:.0f}s")
                return dst
            log("  cache incomplete; rebuilding")
        except Exception as e:
            log("  cache restore failed (", e, "); rebuilding")
    if dst.exists():
        shutil.rmtree(dst)
    build(dst)
    marker.write_text(time.strftime("%Y-%m-%d %H:%M:%S"))
    if keep_cache:
        out = WORK / "cache" / (name + ".tar")
        out.parent.mkdir(parents=True, exist_ok=True)
        part = Path(str(out) + ".part")
        t0 = time.time()
        with tarfile.open(part, "w") as tf:  # uncompressed: JPEGs don't compress further
            tf.add(dst, arcname=dst.name)
        os.replace(part, out)
        log(f"Cached {dst.name} -> {out} ({out.stat().st_size / 2**30:.2f} GB, {time.time() - t0:.0f}s)")
    return dst


def save_decision(topic: str, decision: dict, evidence: dict | None = None) -> Path:
    """Experiment notebooks record what won (and why) for the final training notebooks."""
    p = WORK / "outputs" / f"decision_{topic}.json"
    save_json(dict(topic=topic, decision=decision, evidence=evidence or {},
                   created=time.strftime("%Y-%m-%d %H:%M:%S")), p)
    log(f"DECISION [{topic}]:", decision)
    return p


def load_decision(topic: str) -> dict:
    """Decision from an attached experiment notebook output (or this session); {} if none."""
    name = f"decision_{topic}.json"
    p = WORK / "outputs" / name
    if not p.exists():
        hits = find_files(lambda f: f == name, INPUT_ROOT, max_depth=6, limit=5)
        p = max(hits, key=lambda q: q.stat().st_mtime) if hits else None
    if p is None:
        log(f"No decision_{topic}.json attached: using the notebook defaults.")
        return {}
    d = json.loads(Path(p).read_text()).get("decision", {})
    log(f"Applying experiment decision [{topic}] from {p}: {d}")
    return d


def mark_session(tag: str) -> None:
    """Write a marker so a later session of the SAME notebook can recognise this output."""
    WORK.mkdir(parents=True, exist_ok=True)
    p = WORK / f"st_session_{tag}.json"
    n = json.loads(p.read_text()).get("sessions", 0) if p.exists() else 0
    p.write_text(json.dumps({"tag": tag, "sessions": n + 1, "last": time.strftime("%Y-%m-%d %H:%M:%S")}))


def restore_previous_outputs(tag: str, subdirs=("runs", "outputs", "cache")) -> list[str]:
    """Kaggle resume: copy checkpoints/outputs from an attached previous version of THIS
    notebook (Add Input -> Your Work -> this notebook) into WORK. Recognised by its
    st_session_<tag>.json marker, so other notebooks' outputs are never mixed in."""
    marker = f"st_session_{tag}.json"
    srcs = sorted((Path(dp) for dp, _, fns in walk_limited(INPUT_ROOT, 4) if marker in fns),
                  key=lambda p: (p / marker).stat().st_mtime)
    if not srcs:
        return []
    src = srcs[-1]
    restored = []
    for sub in subdirs:
        if (src / sub).is_dir() and not ((WORK / sub).exists() and any((WORK / sub).iterdir())):
            shutil.copytree(src / sub, WORK / sub, dirs_exist_ok=True)
            restored.append(sub)
    if (src / marker).exists() and not (WORK / marker).exists():
        shutil.copy(src / marker, WORK / marker)
    log(f"Resuming from previous session output {src}: restored {restored}")
    return restored


def run_streaming(cmd: list[str], log_path: Path, stall_min: float = 20.0, env=None,
                  should_stop=None, echo=None, prefix: str = "") -> int:
    """Run a subprocess, mirror its output, kill it if silent for `stall_min` minutes.

    Output is read as raw bytes: every byte (including tqdm carriage-return redraws) resets
    the stall timer, but only completed lines are printed, so a 10 h run does not flood the
    notebook. Returns the exit code (-9 when killed by the stall watchdog)."""
    import signal
    import threading
    log_path.parent.mkdir(parents=True, exist_ok=True)
    posix = hasattr(os, "killpg")
    # New session so the whole process group (DDP spawns workers) can be killed together
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env,
                         start_new_session=posix)
    last = [time.time()]

    def pump():
        buf = b""
        fd = p.stdout.fileno()
        with open(log_path, "ab") as f:
            while True:
                try:
                    chunk = os.read(fd, 65536)
                except OSError:
                    break
                if not chunk:
                    break
                last[0] = time.time()
                f.write(chunk)
                buf += chunk
                *lines, buf = buf.split(b"\n")
                for ln in lines:
                    txt = _progress_line(ln)
                    if txt.strip() and (echo is None or echo(txt)):
                        print(prefix + txt, flush=True)
                if len(buf) > 1_000_000:  # a very long redraw-only stream: keep the tail
                    buf = buf[-10_000:]
            if buf.strip() and (echo is None or echo(_progress_line(buf))):
                print(prefix + _progress_line(buf), flush=True)

    def kill_group():
        if posix:
            try:
                os.killpg(p.pid, signal.SIGKILL)
            except Exception:
                pass

    th = threading.Thread(target=pump, daemon=True)
    th.start()
    while p.poll() is None:
        time.sleep(min(15.0, stall_min * 60 / 4))
        if should_stop is not None and should_stop():
            log("Stop policy triggered (session time budget): stopping training at a checkpoint")
            kill_group()
            p.kill()
            p.wait()
            th.join(timeout=5)
            return STOPPED_BY_POLICY
        if time.time() - last[0] > stall_min * 60:
            log(f"WATCHDOG: no output for {stall_min} min, killing process group {p.pid}")
            kill_group()
            p.kill()
            p.wait()
            th.join(timeout=5)
            return -9
    th.join(timeout=30)
    kill_group()  # reap any DDP worker left behind so the next attempt gets free GPUs
    return p.returncode

In [ ]:
%%writefile /kaggle/working/src/sku110k_prep.py
"""Find SKU-110K under /kaggle/input (any layout) and build a resized YOLO dataset.

Supported sources, tried in order:
  1. Original CSV annotations (annotations_{train,val,test}.csv, no header:
     image,x1,y1,x2,y2,class,image_width,image_height) plus an images folder anywhere.
  2. A YOLO-format copy (images/ + labels/ folders with train/val/test in the path).
  3. Streaming download of the official archive (13.6 GB) that resizes on the fly,
     so the archive itself is never stored.

Images are resized so the long side is `max_side` (SKU-110K photos are ~3000 px and
decoding them is the data-loader bottleneck on Kaggle's 4 CPUs).
"""
from __future__ import annotations

import csv
import os
import random
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from multiprocessing import Pool
from pathlib import Path

import cv2
import numpy as np

from kaggle_common import IMG_EXT, INPUT_ROOT, TMP, disk_free_gb, find_files, log, print_input_tree, walk_limited

OFFICIAL_URL = "http://trax-geometry.s3.amazonaws.com/cvpr_challenge/SKU110K_fixed.tar.gz"
SPLITS = ("train", "val", "test")
cv2.setNumThreads(1)


# ----------------------------------------------------------------------------- CSV parsing
def _is_number(x: str) -> bool:
    try:
        float(x)
        return True
    except ValueError:
        return False


def read_sku_csv(path: Path) -> dict[str, dict]:
    """Returns {image_name: {'w': W, 'h': H, 'boxes': [[x1,y1,x2,y2], ...]}}. Tolerates a header row."""
    out: dict[str, dict] = {}
    with open(path, newline="", encoding="utf-8", errors="replace") as f:
        for row in csv.reader(f):
            if len(row) < 8 or not _is_number(row[1]):
                continue  # header or malformed
            name = os.path.basename(row[0].strip())
            x1, y1, x2, y2 = (float(v) for v in row[1:5])
            w, h = float(row[6]), float(row[7])
            d = out.setdefault(name, {"w": w, "h": h, "boxes": []})
            d["boxes"].append([x1, y1, x2, y2])
    return out


def boxes_to_yolo(boxes, w: float, h: float, min_px: float = 2.0) -> list[str]:
    lines = []
    for x1, y1, x2, y2 in boxes:
        x1, x2 = sorted((max(0.0, min(w, x1)), max(0.0, min(w, x2))))
        y1, y2 = sorted((max(0.0, min(h, y1)), max(0.0, min(h, y2))))
        if x2 - x1 < min_px or y2 - y1 < min_px:
            continue
        lines.append(f"0 {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}")
    return lines


# ----------------------------------------------------------------------------- image IO
def decode_resized(data_or_path, max_side: int, expect_wh: tuple[float, float] | None = None):
    """Decode (fast reduced decode when possible) and resize so the long side <= max_side.

    Returns (image, (orig_w, orig_h)) or (None, reason). Orientation: we try the raw
    pixel order first (EXIF ignored); if the CSV size matches the EXIF-rotated order
    instead, we use that. Boxes in the CSV must match the decoded orientation.
    """
    if isinstance(data_or_path, (str, Path)):
        buf = np.fromfile(str(data_or_path), dtype=np.uint8)
    else:
        buf = np.frombuffer(data_or_path, dtype=np.uint8)
    if buf.size == 0:
        return None, "empty file"

    reduce_flag = cv2.IMREAD_COLOR
    if expect_wh is not None:
        long_side = max(expect_wh)
        if long_side >= 4 * max_side:
            reduce_flag = cv2.IMREAD_REDUCED_COLOR_4
        elif long_side >= 2 * max_side:
            reduce_flag = cv2.IMREAD_REDUCED_COLOR_2
    factor = {cv2.IMREAD_COLOR: 1, cv2.IMREAD_REDUCED_COLOR_2: 2, cv2.IMREAD_REDUCED_COLOR_4: 4}[reduce_flag]

    img = cv2.imdecode(buf, reduce_flag | cv2.IMREAD_IGNORE_ORIENTATION)
    if img is None:
        return None, "decode failed"
    h, w = img.shape[:2]
    orig_w, orig_h = w * factor, h * factor
    if expect_wh is not None:
        ew, eh = expect_wh
        def close(a, b):
            return abs(a - b) <= max(4 * factor, 0.01 * b)
        if not (close(orig_w, ew) and close(orig_h, eh)):
            alt = cv2.imdecode(buf, reduce_flag)  # EXIF-applied orientation
            if alt is not None and close(alt.shape[1] * factor, ew) and close(alt.shape[0] * factor, eh):
                img = alt
                h, w = img.shape[:2]
            elif abs((w / h) - (ew / eh)) > 0.02:
                return None, f"size mismatch {orig_w}x{orig_h} vs csv {ew:.0f}x{eh:.0f}"
            orig_w, orig_h = ew, eh  # same aspect ratio: trust the CSV scale
    s = max_side / max(h, w)
    if s < 1.0:
        img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))), interpolation=cv2.INTER_AREA)
    return img, (orig_w, orig_h)


def _job(args):
    src, dst_img, dst_lbl, boxes, wh, max_side = args
    try:
        if Path(dst_img).exists() and Path(dst_lbl).exists():
            return "skip"
        hint = wh
        if hint is None:
            # YOLO-format source: read the size from the JPEG header (cheap) so decode_resized can use
            # libjpeg's reduced-resolution decode (2-4x faster for ~3000 px SKU-110K photos)
            try:
                from PIL import Image
                with Image.open(src) as im:
                    hint = im.size
            except Exception:
                hint = None
        img, info = decode_resized(src, max_side, hint)
        if img is None:
            return f"bad:{Path(src).name}:{info}"
        w, h = wh if wh else info
        lines = boxes_to_yolo(boxes, w, h) if boxes is not None else None
        if lines is not None and not lines:
            return f"bad:{Path(src).name}:no valid boxes"
        cv2.imwrite(str(dst_img), img, [cv2.IMWRITE_JPEG_QUALITY, 92])
        if lines is not None:
            Path(dst_lbl).write_text("\n".join(lines) + "\n")
        return "ok"
    except Exception as e:  # never let one image kill the pool
        return f"bad:{Path(src).name}:{e}"


# ----------------------------------------------------------------------------- discovery
def index_images(root: Path) -> dict[str, Path]:
    idx: dict[str, Path] = {}
    for dp, _, fns in walk_limited(root, 8):
        for f in fns:
            if Path(f).suffix.lower() in IMG_EXT:
                idx.setdefault(f, Path(dp) / f)
    return idx


def find_csv_source(root: Path = INPUT_ROOT):
    csvs = find_files(lambda f: f.lower().startswith("annotations_") and f.lower().endswith(".csv"), root)
    by_split = {}
    for c in csvs:
        for s in SPLITS:
            if c.stem.lower() == f"annotations_{s}":
                by_split.setdefault(s, c)
    if "train" not in by_split:
        return None
    # The dataset root is the attached input folder that holds the CSVs
    rel = by_split["train"].relative_to(root)
    ds_root = root / rel.parts[0] if len(rel.parts) > 1 else root
    return by_split, ds_root


def find_yolo_source(root: Path = INPUT_ROOT):
    """Find (images_dir, labels_dir) pairs per split in an existing YOLO layout."""
    pairs, counts = {}, {}
    for dp, dns, fns in walk_limited(root, 6):
        p = Path(dp)
        parts = [x.lower() for x in p.parts]
        if "labels" not in parts:
            continue
        n_txt = sum(f.endswith(".txt") for f in fns)
        if n_txt == 0:
            continue
        i = len(parts) - 1 - parts[::-1].index("labels")
        # split folder after (labels/train) or before (train/labels) the 'labels' folder; 'valid' counts as val
        near = parts[i + 1:] + parts[max(0, i - 1):i]
        split = next((s for x in near for s in SPLITS if x == s or x.startswith(s)), None)
        if split is None:
            continue
        img_dir = Path(*p.parts[:i], "images" if p.parts[i] == "labels" else "Images", *p.parts[i + 1:])
        if img_dir.exists() and n_txt > counts.get(split, 0):
            pairs[split], counts[split] = (img_dir, p), n_txt
    return pairs if counts.get("train", 0) >= 20 else None


# ----------------------------------------------------------------------------- builders
def _prepare_out(out: Path):
    for s in SPLITS:
        (out / "images" / s).mkdir(parents=True, exist_ok=True)
        (out / "labels" / s).mkdir(parents=True, exist_ok=True)


def _run_jobs(jobs, workers):
    stats = defaultdict(int)
    bad = []
    with Pool(workers) as pool:
        for i, r in enumerate(pool.imap_unordered(_job, jobs, chunksize=8)):
            key = r.split(":")[0]
            stats[key] += 1
            if key == "bad" and len(bad) < 20:
                bad.append(r)
            if (i + 1) % 1000 == 0:
                log(f"  {i + 1}/{len(jobs)} {dict(stats)}")
    return dict(stats), bad


def build_from_csv(by_split, ds_root, out: Path, max_side: int, limits: dict, workers: int, seed=0):
    log("Indexing images under", ds_root)
    idx = index_images(ds_root)
    log(f"  {len(idx)} image files indexed")
    _prepare_out(out)
    jobs, missing = [], 0
    for s, csv_path in by_split.items():
        ann = read_sku_csv(csv_path)
        names = sorted(ann)
        if limits.get(s):
            random.Random(seed).shuffle(names)
            names = sorted(names[:limits[s]])
        for n in names:
            src = idx.get(n)
            if src is None:
                missing += 1
                continue
            d = ann[n]
            stem = Path(n).stem
            jobs.append((str(src), str(out / "images" / s / f"{stem}.jpg"), str(out / "labels" / s / f"{stem}.txt"),
                         d["boxes"], (d["w"], d["h"]), max_side))
        log(f"  {s}: {len(names)} annotated images")
    if missing:
        log(f"  WARNING: {missing} annotated images not found on disk (skipped)")
    if not jobs:
        raise RuntimeError("CSV annotations found but no matching images. Check the attached dataset.")
    return _run_jobs(jobs, workers)


def build_from_yolo(pairs, out: Path, max_side: int, limits: dict, workers: int, seed=0):
    _prepare_out(out)
    jobs = []
    for s, (img_dir, lbl_dir) in pairs.items():
        ims = sorted(p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXT)
        if limits.get(s):
            random.Random(seed).shuffle(ims)
            ims = ims[:limits[s]]
        for im in ims:
            lbl = lbl_dir / f"{im.stem}.txt"
            if not lbl.exists():
                continue
            dst_lbl = out / "labels" / s / f"{im.stem}.txt"
            # YOLO labels are normalised, so they survive resizing; force class 0 (pack)
            lines = [("0 " + " ".join(l.split()[1:5])) for l in lbl.read_text().splitlines() if len(l.split()) >= 5]
            if not lines:
                continue
            dst_lbl.write_text("\n".join(lines) + "\n")
            jobs.append((str(im), str(out / "images" / s / f"{im.stem}.jpg"), str(dst_lbl), None, None, max_side))
    if "val" not in pairs:  # carve a val split from train
        log("  no val split found: moving 5% of train to val")
    return _run_jobs(jobs, workers)


def build_from_stream(out: Path, max_side: int, limits: dict, url: str = OFFICIAL_URL, threads: int = 4):
    """Stream the official tar.gz, resize every image on the fly, keep only small files."""
    import tarfile
    import urllib.request
    from concurrent.futures import FIRST_COMPLETED, wait
    raw_dir = TMP / "sku_stream"
    (raw_dir / "images").mkdir(parents=True, exist_ok=True)
    (raw_dir / "annotations").mkdir(parents=True, exist_ok=True)

    def work(name, data):
        img, _ = decode_resized(data, max_side, None)
        if img is not None:
            cv2.imwrite(str(raw_dir / "images" / (Path(name).stem + ".jpg")), img, [cv2.IMWRITE_JPEG_QUALITY, 92])

    for attempt in range(2):
        try:
            log(f"Streaming {url} (attempt {attempt + 1}); this takes ~20-40 min")
            with urllib.request.urlopen(url, timeout=120) as resp, \
                    tarfile.open(fileobj=resp, mode="r|gz") as tar, \
                    ThreadPoolExecutor(threads) as ex:
                futs, n = [], 0
                for m in tar:
                    if not m.isfile():
                        continue
                    base = os.path.basename(m.name)
                    if base.lower().endswith(".csv"):
                        (raw_dir / "annotations" / base).write_bytes(tar.extractfile(m).read())
                    elif Path(base).suffix.lower() in IMG_EXT:
                        if (raw_dir / "images" / (Path(base).stem + ".jpg")).exists():
                            continue
                        futs.append(ex.submit(work, base, tar.extractfile(m).read()))
                        n += 1
                        # Backpressure: never hold more than ~32 undecoded images (~100 MB) in RAM
                        while len(futs) > 32:
                            _, pending = wait(futs, return_when=FIRST_COMPLETED)
                            futs = list(pending)
                        if n % 1000 == 0:
                            log(f"  streamed {n} images, disk free {disk_free_gb(TMP):.1f} GB")
            break
        except Exception as e:
            log("stream failed:", e)
            if attempt == 1:
                raise
    # Now we have resized images + original CSVs. Labels are normalised with CSV sizes.
    by_split = {s: raw_dir / "annotations" / f"annotations_{s}.csv" for s in SPLITS
                if (raw_dir / "annotations" / f"annotations_{s}.csv").exists()}
    _prepare_out(out)
    stats = defaultdict(int)
    for s, p in by_split.items():
        ann = read_sku_csv(p)
        names = sorted(ann)
        if limits.get(s):
            random.Random(0).shuffle(names)
            names = names[:limits[s]]
        for n in names:
            src = raw_dir / "images" / (Path(n).stem + ".jpg")
            if not src.exists():
                stats["missing"] += 1
                continue
            d = ann[n]
            lines = boxes_to_yolo(d["boxes"], d["w"], d["h"])
            if not lines:
                stats["bad"] += 1
                continue
            os.replace(src, out / "images" / s / src.name)
            (out / "labels" / s / f"{src.stem}.txt").write_text("\n".join(lines) + "\n")
            stats["ok"] += 1
    return dict(stats), []


def ensure_val_split(out: Path, frac: float = 0.05, seed: int = 0):
    vi = out / "images" / "val"
    if any(vi.glob("*.jpg")):
        return
    ims = sorted((out / "images" / "train").glob("*.jpg"))
    random.Random(seed).shuffle(ims)
    for im in ims[:max(1, int(len(ims) * frac))]:
        os.replace(im, vi / im.name)
        lbl = out / "labels" / "train" / f"{im.stem}.txt"
        if lbl.exists():
            os.replace(lbl, out / "labels" / "val" / lbl.name)


def write_data_yaml(out: Path, names: dict[int, str]) -> Path:
    has_test = any((out / "images" / "test").glob("*.jpg"))
    lines = [f"path: {out}", "train: images/train", "val: images/val"]
    if has_test:
        lines.append("test: images/test")
    lines.append("names:")
    lines += [f"  {k}: {v}" for k, v in names.items()]
    p = out / "data.yaml"
    p.write_text("\n".join(lines) + "\n")
    return p


def count_split(out: Path) -> dict:
    return {s: len(list((out / "images" / s).glob("*.jpg"))) for s in SPLITS}


def prepare_sku110k(out: Path = TMP / "sku110k", max_side: int = 1024, limits: dict | None = None,
                    workers: int | None = None, allow_download: bool = True) -> Path:
    """Main entry. Returns the path of data.yaml. Idempotent: reuses finished work."""
    limits = limits or {}
    workers = workers or max(1, os.cpu_count() or 2)  # all cores: decode+resize is CPU-bound
    yaml_path = out / "data.yaml"
    if yaml_path.exists() and count_split(out)["train"] > 0:
        log("Reusing prepared dataset:", count_split(out))
        return yaml_path

    src = find_csv_source()
    if src:
        by_split, ds_root = src
        log("Found SKU-110K CSV annotations:", {k: str(v) for k, v in by_split.items()})
        stats, bad = build_from_csv(by_split, ds_root, out, max_side, limits, workers)
    else:
        pairs = find_yolo_source()
        if pairs:
            log("Found YOLO-format SKU-110K:", {k: str(v[0]) for k, v in pairs.items()})
            stats, bad = build_from_yolo(pairs, out, max_side, limits, workers)
        elif allow_download:
            print_input_tree()
            log("SKU-110K not found in inputs; falling back to streaming the official archive.")
            if disk_free_gb(TMP) < 8:
                raise RuntimeError("Not enough free disk for the streamed dataset (need ~8 GB).")
            stats, bad = build_from_stream(out, max_side, limits)
        else:
            print_input_tree()
            raise FileNotFoundError("SKU-110K not found. Attach a SKU-110K dataset via 'Add Input'.")
    log("Build stats:", stats)
    for b in bad[:10]:
        log("  ", b)
    ensure_val_split(out)
    counts = count_split(out)
    log("Final split sizes:", counts)
    if counts["train"] < 10 or counts["val"] < 1:
        raise RuntimeError(f"Prepared dataset is too small: {counts}")
    return write_data_yaml(out, {0: "pack"})

In [ ]:
%%writefile /kaggle/working/src/gap_synth.py
"""Build a 2-class (pack, gap) dataset from a prepared SKU-110K YOLO dataset.

Gap labels come from two sources:
  * Real gaps (all splits): in a shelf row (box bottoms clustered), a horizontal space
    between neighbouring packs wider than ~0.6 x the row's median pack width, not
    covered by any other pack. This is a geometric heuristic on real photos.
  * Synthetic gaps (TRAIN split only, spec 5.2.7): erase a run of 1-4 neighbouring packs
    and fill with inpainting, a dark "shelf back" texture, or a real gap patch.

Every image is then cut into square crops resized to 640 px, which matches the
deployment tiling (a 1080p frame split into two ~1080 px tiles, each run at 640).
"""
from __future__ import annotations

import os
import zlib
import random
from multiprocessing import Pool
from pathlib import Path

import cv2
import numpy as np

from kaggle_common import log

PACK, GAP = 0, 1
cv2.setNumThreads(1)


# ----------------------------------------------------------------------------- labels IO
def read_yolo(lbl: Path, w: int, h: int) -> np.ndarray:
    """-> (N,4) xyxy pixels, class 0 only."""
    rows = []
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            p = line.split()
            if len(p) >= 5:
                cx, cy, bw, bh = (float(v) for v in p[1:5])
                rows.append([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h])
    return np.array(rows, dtype=np.float32).reshape(-1, 4)


def write_yolo(path: Path, boxes: np.ndarray, classes: np.ndarray, w: int, h: int):
    lines = []
    for (x1, y1, x2, y2), c in zip(boxes, classes):
        lines.append(f"{int(c)} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}")
    path.write_text("\n".join(lines) + ("\n" if lines else ""))


# ----------------------------------------------------------------------------- geometry
def cluster_rows(boxes: np.ndarray) -> np.ndarray:
    """Row index per box from box bottoms (gap-based 1-D clustering)."""
    if len(boxes) == 0:
        return np.zeros(0, int)
    bottoms = boxes[:, 3]
    med_h = float(np.median(boxes[:, 3] - boxes[:, 1]))
    order = np.argsort(bottoms)
    labels = np.zeros(len(boxes), int)
    row = 0
    for a, b in zip(order[:-1], order[1:]):
        labels[a] = row
        if bottoms[b] - bottoms[a] > 0.5 * med_h:
            row += 1
    labels[order[-1]] = row
    return labels


def covered_fraction(region, boxes: np.ndarray) -> float:
    """Share of `region` covered by the union of `boxes` (coarse raster)."""
    x1, y1, x2, y2 = region
    W, H = max(1, int(x2 - x1)), max(1, int(y2 - y1))
    s = max(1.0, max(W, H) / 64)
    gw, gh = max(1, int(W / s)), max(1, int(H / s))
    grid = np.zeros((gh, gw), bool)
    for bx1, by1, bx2, by2 in boxes:
        ix1 = int(np.clip((bx1 - x1) / s, 0, gw)); ix2 = int(np.clip((bx2 - x1) / s, 0, gw))
        iy1 = int(np.clip((by1 - y1) / s, 0, gh)); iy2 = int(np.clip((by2 - y1) / s, 0, gh))
        if ix2 > ix1 and iy2 > iy1:
            grid[iy1:iy2, ix1:ix2] = True
    return float(grid.mean())


def real_gaps(boxes: np.ndarray, rows: np.ndarray, min_rel: float = 0.6, min_px: float = 12) -> list[list[float]]:
    gaps = []
    for r in np.unique(rows):
        rb = boxes[rows == r]
        if len(rb) < 3:
            continue
        rb = rb[np.argsort(rb[:, 0])]
        med_w = float(np.median(rb[:, 2] - rb[:, 0]))
        for a, b in zip(rb[:-1], rb[1:]):
            space = b[0] - a[2]
            if space > max(min_px, min_rel * med_w):
                g = [float(a[2]), float(min(a[1], b[1])), float(b[0]), float(max(a[3], b[3]))]
                if covered_fraction(g, boxes) < 0.15:
                    gaps.append(g)
    return gaps


# ----------------------------------------------------------------------------- synthesis
def _shelf_back_fill(img, x1, y1, x2, y2, rng):
    h, w = y2 - y1, x2 - x1
    base = np.percentile(img.reshape(-1, 3), 8, axis=0) * rng.uniform(0.5, 1.1)
    grad = np.linspace(rng.uniform(0.55, 0.85), 1.0, h)[:, None, None]  # darker top (shadow)
    patch = base[None, None, :] * grad + rng.normal(0, rng.uniform(2, 7), (h, w, 3))
    patch = cv2.GaussianBlur(np.clip(patch, 0, 255).astype(np.uint8), (5, 5), 0)
    img[y1:y2, x1:x2] = patch


def synth_gaps(img: np.ndarray, boxes: np.ndarray, rows: np.ndarray, rng: random.Random,
               real_gap_patches: list[np.ndarray], max_gaps: int = 2, fill_probs=(0.35, 0.45, 0.20)):
    """Erase runs of packs. Returns (img, kept_pack_boxes, new_gap_boxes).
    fill_probs: probabilities of (inpaint, dark shelf-back texture, real gap patch)."""
    H, W = img.shape[:2]
    keep = np.ones(len(boxes), bool)
    gaps = []
    nrng = np.random.default_rng(rng.randrange(1 << 30))
    row_ids = [r for r in np.unique(rows) if (rows == r).sum() >= 4]
    rng.shuffle(row_ids)
    for r in row_ids[:rng.randint(1, max_gaps)]:
        idx = np.where((rows == r) & keep)[0]
        if len(idx) < 4:
            continue
        idx = idx[np.argsort(boxes[idx, 0])]
        k = rng.randint(1, min(4, len(idx) - 2))
        s = rng.randint(1, len(idx) - k - 1)  # never the row ends: a gap sits between packs
        run = idx[s:s + k]
        x1, y1 = boxes[run, 0].min(), boxes[run, 1].min()
        x2, y2 = boxes[run, 2].max(), boxes[run, 3].max()
        x1, y1 = int(max(0, x1 - 1)), int(max(0, y1 - 1))
        x2, y2 = int(min(W, x2 + 1)), int(min(H, y2 + 1))
        if x2 - x1 < 8 or y2 - y1 < 8:
            continue
        mode = rng.random()
        p_inp, p_back = fill_probs[0], fill_probs[0] + fill_probs[1]
        if mode < p_inp:
            m = np.zeros((H, W), np.uint8)
            m[y1:y2, x1:x2] = 255
            # inpaint only a local window: fast and uses nearby shelf texture
            px, py = (x2 - x1), (y2 - y1)
            wx1, wy1 = max(0, x1 - px), max(0, y1 - py // 2)
            wx2, wy2 = min(W, x2 + px), min(H, y2 + py // 2)
            win = cv2.inpaint(img[wy1:wy2, wx1:wx2], m[wy1:wy2, wx1:wx2], 5, cv2.INPAINT_TELEA)
            img[wy1:wy2, wx1:wx2] = win
        elif mode < p_back or not real_gap_patches:
            _shelf_back_fill(img, x1, y1, x2, y2, nrng)
        else:
            patch = real_gap_patches[rng.randrange(len(real_gap_patches))]
            img[y1:y2, x1:x2] = cv2.resize(patch, (x2 - x1, y2 - y1), interpolation=cv2.INTER_LINEAR)
        keep[run] = False
        # Drop other packs that were mostly erased
        for j in np.where(keep)[0]:
            bx1, by1, bx2, by2 = boxes[j]
            inter = max(0, min(bx2, x2) - max(bx1, x1)) * max(0, min(by2, y2) - max(by1, y1))
            if inter > 0.3 * (bx2 - bx1) * (by2 - by1):
                keep[j] = False
        gaps.append([x1, y1, x2, y2])
    return img, boxes[keep], gaps


# ----------------------------------------------------------------------------- crops
def random_crops(H: int, W: int, n: int, rng: random.Random, lo=0.55, hi=0.9) -> list[tuple[int, int, int, int]]:
    out = []
    short = min(H, W)
    for _ in range(n):
        s = int(short * rng.uniform(lo, hi))
        x = rng.randint(0, W - s)
        y = rng.randint(0, H - s)
        out.append((x, y, x + s, y + s))
    return out


def clip_boxes(boxes: np.ndarray, crop, min_vis: float = 0.45) -> tuple[np.ndarray, np.ndarray]:
    if len(boxes) == 0:
        return np.zeros((0, 4), np.float32), np.zeros(0, bool)
    cx1, cy1, cx2, cy2 = crop
    b = boxes.copy()
    area = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    b[:, [0, 2]] = np.clip(b[:, [0, 2]], cx1, cx2)
    b[:, [1, 3]] = np.clip(b[:, [1, 3]], cy1, cy2)
    vis = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1]) / np.maximum(area, 1e-6)
    keep = (vis >= min_vis) & (b[:, 2] - b[:, 0] >= 3) & (b[:, 3] - b[:, 1] >= 3)
    b = b[keep]
    b[:, [0, 2]] -= cx1
    b[:, [1, 3]] -= cy1
    return b, keep


def _process(args):
    img_path, lbl_path, out_dir, split, seed, n_crops, out_size, p_synth, fill_probs = args
    try:
        rng = random.Random(seed)
        img = cv2.imread(img_path)
        if img is None:
            return dict(status="bad")
        H, W = img.shape[:2]
        packs = read_yolo(Path(lbl_path), W, H)
        if len(packs) < 3:
            return dict(status="few")
        rows = cluster_rows(packs)
        gaps = real_gaps(packs, rows)
        n_real = len(gaps)
        n_syn = 0
        if split == "train" and rng.random() < p_synth:
            patches = []
            for g in gaps:
                x1, y1, x2, y2 = (int(v) for v in g)
                if x2 - x1 > 4 and y2 - y1 > 4:
                    patches.append(img[y1:y2, x1:x2].copy())
            img, packs, syn = synth_gaps(img, packs, rows, rng, patches, fill_probs=fill_probs)
            gaps += syn
            n_syn = len(syn)
        gaps_a = np.array(gaps, np.float32).reshape(-1, 4)
        stem = Path(img_path).stem
        crops = random_crops(H, W, n_crops, rng) if split == "train" else [_center_crop(H, W)]
        for ci, crop in enumerate(crops):
            pb, _ = clip_boxes(packs, crop)
            gb, _ = clip_boxes(gaps_a, crop, min_vis=0.6)
            s = crop[2] - crop[0]
            sc = out_size / s
            im = cv2.resize(img[crop[1]:crop[3], crop[0]:crop[2]], (out_size, out_size),
                            interpolation=cv2.INTER_AREA if sc < 1 else cv2.INTER_LINEAR)
            name = f"{stem}_c{ci}"
            cv2.imwrite(str(Path(out_dir) / "images" / split / f"{name}.jpg"), im, [cv2.IMWRITE_JPEG_QUALITY, 92])
            boxes = np.concatenate([pb, gb]) * sc
            cls = np.concatenate([np.full(len(pb), PACK), np.full(len(gb), GAP)])
            write_yolo(Path(out_dir) / "labels" / split / f"{name}.txt", boxes, cls, out_size, out_size)
        return dict(status="ok", real=n_real, syn=n_syn)
    except Exception as e:
        return dict(status="bad", err=str(e))


def _center_crop(H, W):
    """Val/test: one deterministic crop covering most of the short side."""
    s = int(min(H, W) * 0.9)
    x, y = (W - s) // 2, (H - s) // 2
    return (x, y, x + s, y + s)


def build_pack_gap(src_root: Path, out: Path, limits: dict | None = None, n_crops: int = 2,
                   out_size: int = 640, p_synth: float = 0.6, workers: int | None = None, seed: int = 0,
                   fill_probs=(0.35, 0.45, 0.20)) -> Path:
    limits = limits or {}
    if (out / "data.yaml").exists():
        log("Reusing", out)
        return out / "data.yaml"
    jobs = []
    for split in ("train", "val", "test"):
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)
        ims = sorted((src_root / "images" / split).glob("*.jpg"))
        if limits.get(split):
            random.Random(seed).shuffle(ims)
            ims = ims[:limits[split]]
        for i, im in enumerate(ims):
            jobs.append((str(im), str(src_root / "labels" / split / f"{im.stem}.txt"), str(out), split,
                         seed * 1_000_003 + zlib.crc32(im.name.encode()) % 1_000_003, n_crops, out_size, p_synth, tuple(fill_probs)))
    stats = dict(ok=0, bad=0, few=0, real=0, syn=0)
    with Pool(workers or max(1, os.cpu_count() or 2)) as pool:
        for i, r in enumerate(pool.imap_unordered(_process, jobs, chunksize=4)):
            stats[r["status"]] += 1
            stats["real"] += r.get("real", 0)
            stats["syn"] += r.get("syn", 0)
            if (i + 1) % 1000 == 0:
                log(f"  {i + 1}/{len(jobs)} {stats}")
    log("pack+gap build:", stats)
    if stats["ok"] == 0:
        raise RuntimeError("No pack+gap images were produced.")
    has_test = any((out / "images" / "test").glob("*.jpg"))
    yaml = [f"path: {out}", "train: images/train", "val: images/val"] + (["test: images/test"] if has_test else [])
    yaml += ["names:", "  0: pack", "  1: gap"]
    (out / "data.yaml").write_text("\n".join(yaml) + "\n")
    return out / "data.yaml"


# ----------------------------------------------------------------------------- metric
def gap_recall_at_precision(weights: Path, data_root: Path, split: str = "test", target_p: float = 0.9,
                            iou_thr: float = 0.5, imgsz: int = 640, max_images: int = 2000) -> dict:
    """Gap recall at a fixed precision (spec 6.3), by sweeping the confidence threshold."""
    import torch
    from ultralytics import YOLO
    model = YOLO(str(weights))
    ims = sorted((data_root / "images" / split).glob("*.jpg"))[:max_images]
    if not ims:
        return {}
    dev = 0 if torch.cuda.is_available() else "cpu"
    scores, tps, n_gt = [], [], 0
    for i in range(0, len(ims), 32):
        batch = ims[i:i + 32]
        res = model.predict([str(p) for p in batch], imgsz=imgsz, conf=0.01, iou=0.6, max_det=1000,
                            device=dev, verbose=False, half=dev != "cpu")
        for p, r in zip(batch, res):
            h, w = r.orig_shape
            gt = []
            lbl = data_root / "labels" / split / f"{p.stem}.txt"
            for line in (lbl.read_text().splitlines() if lbl.exists() else []):
                q = line.split()
                if len(q) >= 5 and int(q[0]) == GAP:
                    cx, cy, bw, bh = (float(v) for v in q[1:5])
                    gt.append([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h])
            gt = np.array(gt).reshape(-1, 4)
            n_gt += len(gt)
            if r.boxes is None or len(r.boxes) == 0:
                continue
            cls = r.boxes.cls.cpu().numpy().astype(int)
            pb = r.boxes.xyxy.cpu().numpy()[cls == GAP]
            pc = r.boxes.conf.cpu().numpy()[cls == GAP]
            used = np.zeros(len(gt), bool)
            for j in np.argsort(-pc):
                tp = False
                if len(gt):
                    ious = _iou(pb[j], gt)
                    ious[used] = 0
                    k = int(np.argmax(ious))
                    if ious[k] >= iou_thr:
                        used[k] = True
                        tp = True
                scores.append(float(pc[j]))
                tps.append(tp)
    if not scores or n_gt == 0:
        return dict(n_gt=n_gt, recall_at_p=0.0, conf_at_p=None)
    order = np.argsort(-np.array(scores))
    tp = np.cumsum(np.array(tps)[order])
    fp = np.cumsum(~np.array(tps)[order])
    prec = tp / np.maximum(tp + fp, 1)
    rec = tp / n_gt
    ok = np.where(prec >= target_p)[0]
    if not len(ok):
        return dict(n_gt=n_gt, recall_at_p=0.0, conf_at_p=None, max_precision=float(prec.max()))
    j = ok[np.argmax(rec[ok])]
    return dict(n_gt=int(n_gt), target_precision=target_p, recall_at_p=float(rec[j]),
                conf_at_p=float(np.array(scores)[order][j]), images=len(ims))


def _iou(b, g):
    x1 = np.maximum(b[0], g[:, 0]); y1 = np.maximum(b[1], g[:, 1])
    x2 = np.minimum(b[2], g[:, 2]); y2 = np.minimum(b[3], g[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    a = (b[2] - b[0]) * (b[3] - b[1]); ag = (g[:, 2] - g[:, 0]) * (g[:, 3] - g[:, 1])
    return inter / (a + ag - inter + 1e-9)

In [ ]:
%%writefile /kaggle/working/src/yolo_runner.py
"""Resumable, fast Ultralytics training for Kaggle and Colab free tiers.

* Runs as a subprocess script (Ultralytics multi-GPU DDP is fragile inside Jupyter).
* Uses both T4s on Kaggle (DDP); falls back to one GPU if DDP fails.
* Fixed-epoch cosine schedule spanning sessions: each session trains until its time budget,
  stops right AFTER an epoch checkpoint is written, and the next session resumes the same run
  (`resume=True`) from last.pt. Checkpoints live in WORK (Kaggle output / Google Drive).
* Stall watchdog kills a hung process group; corrupted last.pt falls back to best.pt.
"""
from __future__ import annotations

import os
import shutil
import sys
import time
from pathlib import Path

from kaggle_common import (STOPPED_BY_POLICY, WORK, cpu_workers, elapsed_h, gpu_info, log, run_streaming,
                           save_json)

HERE = Path(__file__).resolve().parent
CKPT_SETTLE_S = (10.0, 120.0)  # a checkpoint is "complete and recent" in this age window


def run_dir(name: str) -> Path:
    return WORK / "runs" / name


def _find_run_file(run_name: str, fname: str) -> Path | None:
    p = run_dir(run_name) / ("weights" if fname.endswith(".pt") else "") / fname
    if p.exists():
        return p
    c = [q for q in WORK.rglob(fname) if run_name in q.parts]
    return max(c, key=lambda q: q.stat().st_mtime) if c else None


def ckpt_epoch(path: Path) -> int | None:
    """Epoch stored in an Ultralytics checkpoint: -1 once training finished (optimizer stripped),
    else the last completed epoch; None if unreadable (e.g. cut mid-write)."""
    try:
        import torch
        ck = torch.load(str(path), map_location="cpu", weights_only=False)  # our own Ultralytics file
        return int(ck.get("epoch", -1))
    except Exception as e:
        log(f"cannot read {path}: {e}")
        return None


def epochs_done(name: str) -> int:
    r = run_dir(name) / "results.csv"
    if not r.exists():
        return 0
    return max(0, len([l for l in r.read_text().splitlines() if l.strip()]) - 1)


def _stop_policy(name: str, deadline_ts: float):
    """Stop when the next epoch would not finish before the deadline, right after last.pt is saved."""
    last = run_dir(name) / "weights" / "last.pt"
    state = dict(mtimes=[], seen=None)

    def should_stop() -> bool:
        now = time.time()
        if now >= deadline_ts:
            return True
        if not last.exists():
            return False
        m = last.stat().st_mtime
        if m != state["seen"]:
            state["seen"] = m
            state["mtimes"].append(m)
            return False  # just written; let the write finish
        if len(state["mtimes"]) >= 2:
            ep = (state["mtimes"][-1] - state["mtimes"][0]) / (len(state["mtimes"]) - 1)
        else:
            ep = None
        stable = CKPT_SETTLE_S[0] <= now - m <= CKPT_SETTLE_S[1]  # checkpoint complete and recent
        return bool(ep and stable and now + 1.15 * ep > deadline_ts)

    return should_stop


def train(train_args: dict, budget_h: float, multi_gpu: bool = True, stall_min: float = 25.0,
          device: str | None = None, echo=None) -> dict:
    """Train (or resume) run `train_args['name']` for at most `budget_h` hours this session.

    device: force a device (e.g. "1" to run an experiment on the second GPU in parallel).
    echo:   optional line filter for the console (full output always goes to logs/<name>.log).
    Returns dict(best=Path|None, last=Path|None, finished=bool, epochs_done=int)."""
    name = train_args["name"]
    rd = run_dir(name)
    w = rd / "weights"
    deadline = time.time() + budget_h * 3600
    gpus = gpu_info()
    env = dict(os.environ)
    env["PYTHONPATH"] = str(HERE) + os.pathsep + env.get("PYTHONPATH", "")
    if device is not None:
        use_ddp = "," in device
    else:
        log("GPUs:", gpus)
        use_ddp = multi_gpu and len(gpus) >= 2
        device = "0,1" if use_ddp else ("0" if gpus else "cpu")
    batch = int(train_args.get("batch", 16))
    if not use_ddp and train_args.get("batch_per_gpu"):
        batch = int(train_args["batch_per_gpu"])
    workers = train_args.get("workers") or max(2, cpu_workers() // (2 if use_ddp else 1))

    restart_from, restarted = None, False
    for attempt in range(5):
        if time.time() > deadline - min(120.0, 0.1 * budget_h * 3600):
            log("No session time left for training.")
            break
        last, best = w / "last.pt", w / "best.pt"
        ep = ckpt_epoch(last) if last.exists() else None
        if ep is None and last.exists() and best.exists():
            be = ckpt_epoch(best)
            if be is not None and be >= 0:
                log("last.pt unreadable; resuming from best.pt instead")
                shutil.copy(best, last)
                ep = be
        if ep == -1:
            log(f"Run '{name}' already finished.")
            break
        if restart_from is not None:
            # resume kept failing (e.g. GPU count changed): new run initialised from the old weights
            init = WORK / "runs" / f"{name}_init.pt"
            shutil.copy(restart_from, init)
            remaining = max(1, int(train_args.get("epochs", 50)) - epochs_done(name))
            shutil.rmtree(rd, ignore_errors=True)
            cfg = {k: v for k, v in train_args.items() if k != "batch_per_gpu"}
            cfg.update(model=str(init), epochs=remaining, project=str(WORK / "runs"), exist_ok=True,
                       device=device, batch=batch, workers=workers, warmup_epochs=0)
            log(f"Restarting '{name}' from saved weights for the remaining {remaining} epochs")
            restart_from = None
        elif ep is not None and ep >= 0:
            # resume keeps the original schedule; device/batch may change between sessions
            cfg = dict(model=str(last), resume=True, device=device, batch=batch, workers=workers)
            log(f"Resuming '{name}' after epoch {ep + 1} on device {device}")
        else:
            if rd.exists():
                shutil.rmtree(rd)
            cfg = {k: v for k, v in train_args.items() if k != "batch_per_gpu"}
            cfg.update(project=str(WORK / "runs"), exist_ok=True, device=device, batch=batch, workers=workers)
            if device == "cpu":
                log("WARNING: no GPU visible; enable a GPU accelerator. Training on CPU is very slow.")
                cfg["amp"] = False
        if device == "0,1" and cfg["batch"] % 2:
            cfg["batch"] += 1
        cfg_path = WORK / f"train_cfg_{name}.json"
        save_json(cfg, cfg_path)
        before = epochs_done(name)
        rc = run_streaming([sys.executable, str(HERE / "train_yolo_cli.py"), str(cfg_path)],
                           WORK / "logs" / f"{name}.log", stall_min=stall_min, env=env,
                           should_stop=_stop_policy(name, deadline), echo=echo,
                           prefix=f"[{name}] " if echo is not None else "")
        _free_gpu()
        progressed = epochs_done(name) > before
        log(f"attempt {attempt + 1}: exit {rc}, epochs done {epochs_done(name)} (+{epochs_done(name) - before})")
        if rc == STOPPED_BY_POLICY:
            break
        if rc == 0:
            continue  # loop re-reads last.pt: -1 means finished
        if device == "0,1" and not progressed:
            log("Multi-GPU run failed without progress; falling back to a single GPU")
            device = "0"
            batch = int(train_args.get("batch_per_gpu") or max(2, batch // 2))
            workers = train_args.get("workers") or cpu_workers()
        elif not progressed and last.exists() and not restarted:
            restart_from, restarted = last, True
        elif not progressed:
            break
    last, best = w / "last.pt", w / "best.pt"
    finished = last.exists() and ckpt_epoch(last) == -1
    res = dict(best=best if best.exists() else (last if last.exists() else None),
               last=last if last.exists() else None, finished=finished, epochs_done=epochs_done(name))
    log("Training status:", {k: str(v) for k, v in res.items()})
    if res["best"] is None:
        raise RuntimeError(f"No checkpoint produced for '{name}'. See {WORK / 'logs' / (name + '.log')}")
    if not finished:
        log("NOT FINISHED: run this notebook again (Kaggle: attach this version's output as input; "
            "Colab: just Run all) to resume from the last epoch.")
    return res


def _free_gpu():
    try:
        import gc
        import torch
        gc.collect()
        torch.cuda.empty_cache()
    except Exception:
        pass
    time.sleep(3)


def evaluate(weights: Path, data_yaml: Path, imgsz: int, split: str = "test", max_det: int = 1000,
             iou: float = 0.6, batch: int = 16) -> dict:
    from ultralytics import YOLO
    import torch
    dev = 0 if torch.cuda.is_available() else "cpu"
    m = YOLO(str(weights))
    try:
        r = m.val(data=str(data_yaml), split=split, imgsz=imgsz, batch=batch, max_det=max_det, iou=iou,
                  conf=0.001, device=dev, half=dev != "cpu", plots=True, project=str(WORK / "runs"),
                  name=f"eval_{split}", exist_ok=True, verbose=True, workers=cpu_workers())
    except Exception as e:
        if split != "val":
            log(f"Eval on '{split}' failed ({e}); falling back to 'val'")
            return evaluate(weights, data_yaml, imgsz, "val", max_det, iou, batch)
        raise
    out = dict(split=split, mAP50=float(r.box.map50), mAP50_95=float(r.box.map),
               precision=float(r.box.mp), recall=float(r.box.mr))
    try:
        names = r.names
        out["per_class_AP50"] = {names[int(c)]: float(ap) for c, ap in zip(r.box.ap_class_index, r.box.ap50)}
    except Exception:
        pass
    log("Eval:", out)
    return out


def export_onnx(weights: Path, imgsz: int) -> Path | None:
    try:
        from ultralytics import YOLO
        p = YOLO(str(weights)).export(format="onnx", dynamic=True, simplify=True, imgsz=imgsz)
        return Path(p)
    except Exception as e:
        log("ONNX export failed (non-fatal):", e)
        return None


def package_outputs(files: dict[str, Path | None], out_dir: Path = WORK / "outputs") -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    for dst, src in files.items():
        if src is not None and Path(src).exists() and Path(src).resolve() != (out_dir / dst).resolve():
            shutil.copy2(src, out_dir / dst)
            log("saved", out_dir / dst)
    return out_dir

In [ ]:
%%writefile /kaggle/working/src/train_yolo_cli.py
"""Run one Ultralytics training from a JSON config (called as a subprocess)."""
import json
import sys

from kaggle_common import quiet_loggers

if __name__ == "__main__":
    quiet_loggers()
    cfg = json.load(open(sys.argv[1]))
    from ultralytics import YOLO
    dev = cfg.get("device")
    if isinstance(dev, str) and "," in dev:
        cfg["device"] = [int(x) for x in dev.split(",")]
    model = YOLO(cfg.pop("model"))
    model.train(**cfg)

In [ ]:
import kaggle_common as kc, sku110k_prep as sp, gap_synth as gs, yolo_runner as yr
kc.quiet_loggers()
kc.restore_previous_outputs("nb02")
kc.mark_session("nb02")
init = kc.find_weights(("pack_detector_sku110k.pt",)) or kc.find_weights(("best.pt",), must_contain="sku110k")
kc.log("GPUs:", kc.gpu_info(), "| CPUs:", os.cpu_count(), "| initial weights:", init or "none -> yolo11s.pt (COCO)")

In [ ]:
sfx = "_smoke" if SMOKE_TEST else ""
src_name = f"sku110k_{SRC_SIDE}{sfx}"
lim = dict(train=40, val=10, test=10) if SMOKE_TEST else {}
src_dir = kc.cached_dir(src_name, kc.TMP / src_name, lambda d: sp.prepare_sku110k(out=d, max_side=SRC_SIDE, limits=lim),
                        keep_cache=False)   # notebook 01's cache is reused when attached
pg_name = f"packgap_{IMGSZ}_{MAX_TRAIN_IMAGES}{sfx}"
pg_dir = kc.cached_dir(pg_name, kc.TMP / pg_name, lambda d: gs.build_pack_gap(
    src_dir, d, limits=dict(train=24) if SMOKE_TEST else dict(train=MAX_TRAIN_IMAGES), n_crops=2, out_size=IMGSZ, p_synth=0.6))
pg_yaml = pg_dir / "data.yaml"
print(open(pg_yaml).read())

In [ ]:
# Preview a few training crops with labels (sanity check of the synthetic gaps)
import cv2, random, matplotlib.pyplot as plt
ims = sorted((pg_dir / "images" / "train").glob("*.jpg")); random.Random(1).shuffle(ims)
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for ax, p in zip(axs, ims[:4]):
    im = cv2.imread(str(p))[:, :, ::-1].copy(); h, w = im.shape[:2]
    for line in (pg_dir / "labels" / "train" / f"{p.stem}.txt").read_text().splitlines():
        c, cx, cy, bw, bh = line.split(); cx, cy, bw, bh = map(float, (cx, cy, bw, bh))
        cv2.rectangle(im, (int((cx - bw / 2) * w), int((cy - bh / 2) * h)), (int((cx + bw / 2) * w), int((cy + bh / 2) * h)),
                      (255, 0, 0) if c == "1" else (0, 200, 0), 2 if c == "1" else 1)
    ax.imshow(im); ax.axis("off")
plt.suptitle("green = pack, red = gap"); plt.savefig("/kaggle/working/packgap_preview.png", dpi=80); plt.show()

In [ ]:
train_args = dict(
    model=str(init) if init else "yolo11s.pt", data=str(pg_yaml), imgsz=IMGSZ, epochs=2 if SMOKE_TEST else EPOCHS,
    batch=8 if SMOKE_TEST else BATCH, batch_per_gpu=4 if SMOKE_TEST else BATCH_PER_GPU, name="packgap" + sfx,
    optimizer="AdamW", lr0=0.001, cos_lr=True, warmup_epochs=1,
    mosaic=0.5, close_mosaic=0 if SMOKE_TEST else 5, fliplr=0.5, degrees=0.0, hsv_v=0.5,
    max_det=1000, amp=True, patience=12, seed=0, deterministic=False,
)
res = yr.train(train_args, budget_h=SESSION_HOURS - kc.elapsed_h() - RESERVE_H)

In [ ]:
split = "test" if res["finished"] and (pg_dir / "images" / "test").exists() else "val"
metrics = yr.evaluate(res["best"], pg_yaml, imgsz=IMGSZ, split=split, max_det=1000, iou=0.6)
gap = gs.gap_recall_at_precision(res["best"], pg_dir, split=split, target_p=0.9, imgsz=IMGSZ,
                                 max_images=200 if SMOKE_TEST else 2000)
print("gap recall @ precision 0.9:", gap)
onnx = yr.export_onnx(res["best"], IMGSZ) if res["finished"] else None
out = yr.package_outputs({"pack_gap.pt": res["best"], "pack_gap.onnx": onnx,
                          "results_02.csv": yr.run_dir(train_args["name"]) / "results.csv",
                          "packgap_preview.png": kc.WORK / "packgap_preview.png"})
kc.save_json(dict(metrics=metrics, gap_recall_at_p90=gap, init_weights=str(init), finished=res["finished"],
                  epochs_done=res["epochs_done"], smoke_test=SMOKE_TEST, hours_used=round(kc.elapsed_h(), 2)),
             out / "metrics_02.json")
print("ALL DONE" if res["finished"] else "NOT FINISHED: attach this version's output as input and run again to resume.")